# Data Merging

Build the full item-level dataset from the registered order-items, orders, customers, products and sellers sources. Each row represents one order item, identified by `(order_id, order_item_id)`.

Run the cells from top to bottom. The merge step saves `merged_order_items` in `data/interim`; subsequent cells display merge diagnostics and check the reference dataset. Reference dimensions are **112,650 rows, 25 columns and 98,666 orders**.

For other table combinations and storage options, see the [Merge User Guide](../docs/Merge/User_Guide.md). Function contracts and implementation details are in the [Merge Technical Reference](../docs/Merge/Technical_Reference.md).

## 1. Input and execution context

Locate the repository from the current directory or its parents, then import the shared paths and merge interfaces. Source CSV files are read from `data/raw`; merged output belongs in `data/interim`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


# Find the project root from the current directory
def find_project_root(start_path: Path) -> Path:
    """Find project root by locating the src/ directory and requirements.txt."""
    for path in [start_path, *start_path.parents]:
        if (path / "src").is_dir() and (path / "requirements.txt").is_file():
            return path

    raise FileNotFoundError(
        "Could not locate project root containing both 'src/' and 'requirements.txt'."
    )


PROJECT_ROOT = find_project_root(Path.cwd())


# Ensure Python recognizes the 'src' package
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# Import project paths
from src.paths import RAW_DIR, INTERIM_DIR


# Import pipeline modules
from src.pipeline.api import merge_data
from src.pipeline.contracts.schema import BENCHMARK_ROW_COUNT, FULL_MERGE_SCHEMA
from src.pipeline.contracts.validator import validate_table_contract
from src.pipeline.merging.planner import MERGE_TABLE_SPECS
from src.pipeline.merging.reporting import build_merge_report_tables
from src.pipeline.run_merge import save_merged_dataset


# Check the environment
print(f"Project root: {PROJECT_ROOT}")
print(f"Data raw path: {RAW_DIR}")
print(f"Interim data path: {INTERIM_DIR}")

## 2. Full merge and persistence

Merge all five registered source tables using the shared API. The result includes the item table, its schema and an in-memory merge report. Save the full dataset with Parquet preferred and CSV fallback; `saved_path` records the successful format. The preview retains original data values.

In [ ]:
# Merge and validate the full registered dataset.
merge_result = merge_data(tables=list(MERGE_TABLE_SPECS))
df_merged = merge_result.data

# Save to interim, trying Parquet first with CSV fallback.
saved_path = save_merged_dataset(df_merged, output_dir=INTERIM_DIR)

# Quick check of dimensions and data
print(f"Final shape: {df_merged.shape}")
df_merged.head(3)

## 3. Merge diagnostics

Display the eight report tables: overview, sources, joins, row counts, key checks, column selection, validation and schema. These tables describe the current merge, including join relationships and recorded checks. They are displayed in the notebook and do not update documentation.

In [ ]:
# Display output structure and checks recorded during merging.
merge_report_tables = build_merge_report_tables(merge_result)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    for section, table in merge_report_tables.items():
        print(f"\n{section.replace('_', ' ').title()}")
        if table.empty:
            print("No records reported.")
        else:
            display(table)

## 4. Contract and reference validation

Validate canonical columns, dtypes, unique item keys and non-null requirements. Compare dimensions and order-item preservation with the current raw-data reference. Repeated `order_id` values are expected because an order may contain several items. A reference mismatch raises an error for investigation.

In [ ]:
# Reference checks for the full item-level output of the current raw dataset.
base_rows = merge_result.report.get("input_shapes", {}).get("order_items", {}).get("rows")
if merge_result.schema != FULL_MERGE_SCHEMA or base_rows is None:
    raise ValueError("Run this baseline cell on the full Olist merge with its order_items report.")

# Reuse the shared column, dtype, unique-key and non-null checks.
validate_table_contract(df_merged, FULL_MERGE_SCHEMA, table_name="full_merge")

baseline_checks = pd.DataFrame(
    [
        ["Rows", BENCHMARK_ROW_COUNT, len(df_merged)],
        ["Columns", len(FULL_MERGE_SCHEMA.columns), len(df_merged.columns)],
        ["Unique orders", 98_666, df_merged["order_id"].nunique()],
        ["Order-item rows preserved", base_rows, len(df_merged)],
    ],
    columns=["Check", "Expected", "Actual"],
)
baseline_checks["Passed"] = baseline_checks["Expected"].eq(baseline_checks["Actual"])
display(baseline_checks)

if not baseline_checks["Passed"].all():
    failed_checks = ", ".join(baseline_checks.loc[~baseline_checks["Passed"], "Check"])
    raise ValueError(
        f"Full merge baseline mismatch: {failed_checks}. "
        "Check the raw dataset version and merge inputs against these reference counts."
    )

print("Full merge contract and baseline checks passed.")